# Modelagem — uplift de cupons iFood

Implementa `specification/02-modeling/spec.md` a partir do dataset processado
(`data/processed/`, escrito por `python -m src.pipeline`). Toda lógica vive em
`src/` — este notebook importa, executa e exibe, na ordem em que a spec 02 pede:
split temporal → baseline preditivo → X-learner de uplift → avaliação Qini/AUUC.

Cada seção roda, mostra o resultado e fecha com uma leitura curta antes da
próxima. Passos ainda não implementados (T-206 em diante: política, IPW,
impacto em R$, SHAP) ficam marcados como próximo passo, não como célula vazia.

**Pré-requisito:** `data/processed/` precisa existir (`uv run python -m src.pipeline`).

## 0. Setup

Raiz do repo por ancestralidade de `config.yaml` (sem caminho literal). Lê o
dataset processado direto — nenhuma transformação de pipeline roda aqui.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import pandas as pd
from pyspark.sql import functions as F

from src import eda, model_baseline, split, uplift, uplift_eval, viz
from src.attribution import attribute
from src.config import load
from src.io import parse_events, read_offers
from src.pipeline import build_spark

pd.set_option("display.max_columns", None)

cfg = load()
spark = build_spark(cfg, app_name="ifood-uplift-modeling")
processed = spark.read.parquet(str(cfg.processed_dir))
print(f"{processed.count():,} linhas no dataset processado.")

/home/caioolubini/Projects/ifood-coupons-uplift/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/09 22:21:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


76,277 linhas no dataset processado.


## 1. Split temporal

Treino = ondas `[0, cfg.validation_wave_cutoff)`; holdout = ondas
`[cfg.validation_wave_cutoff, cfg.n_campaign_waves)`. Nunca split aleatório —
`assert_temporal_order` barra qualquer corte que inverta a ordem no tempo. O
mesmo cliente pode aparecer nos dois lados (recebeu ofertas em ondas
diferentes); isso é esperado, não é vazamento.

In [2]:
train_sdf, holdout_sdf = split.temporal_split(processed, cfg)
split.assert_temporal_order(train_sdf, holdout_sdf)

train_df = train_sdf.toPandas()
holdout_df = holdout_sdf.toPandas()

resumo_split = pd.DataFrame([
    {"lado": "treino", "ondas": f"[0, {cfg.validation_wave_cutoff})",
     "linhas": len(train_df), "taxa_conversao": train_df["converted"].mean()},
    {"lado": "holdout", "ondas": f"[{cfg.validation_wave_cutoff}, {cfg.n_campaign_waves})",
     "linhas": len(holdout_df), "taxa_conversao": holdout_df["converted"].mean()},
])
resumo_split

26/07/09 22:21:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,lado,ondas,linhas,taxa_conversao
0,treino,"[0, 4)",50808,0.363663
1,holdout,"[4, 6)",25469,0.305037


Treino e holdout devem ter volume e taxa de conversão na mesma ordem de
grandeza — divergência grande aqui sinalizaria censura à direita concentrada
nas últimas ondas (divergência #4 registrada em `CLAUDE.md`), não um bug de split.

## 2. Baseline preditivo (logística + LGBM)

Não é o modelo de uplift — é o degrau anterior que prova que há sinal
aprendível em `converted` antes de pagar a complexidade extra do X-learner
(Premissa 5). Seleção seguinte (§3-4) nunca vai usar AUC; aqui é a métrica
certa porque a pergunta é outra: "dá para prever completion, e faz sentido
gastar o X-learner?".

In [3]:
logit, lgbm, baseline_metrics = model_baseline.train_and_log(train_df, holdout_df, cfg)
pd.Series(baseline_metrics, name="AUC (holdout)").to_frame()

2026/07/09 22:21:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/09 22:21:27 INFO mlflow.store.db.utils: Updating database tables


2026/07/09 22:21:28 INFO mlflow.tracking.fluent: Experiment with name 'ifood-uplift' does not exist. Creating a new experiment.


,AUC (holdout)
auc_logit,0.745709
auc_lgbm,0.820885


`auc_lgbm >= auc_logit` é o aceite de T-203: se a árvore não bate a âncora
linear, há um bug a investigar antes de seguir. Run registrado em MLflow
(`cfg.mlflow_tracking_uri`, experimento `cfg.mlflow_experiment_name`).

## 3. X-learner de uplift

Um X-learner por `offer_type` (bogo, discount, informational) — cada tipo tem
mecanismo de recompensa distinto, então cada um aprende sua própria função de
efeito. `treatment` = viu a oferta (exposição real, não recebimento);
`converted` é o outcome. Treina no lado de treino do split.

O X-learner é τ(x) = p·τ_c(x) + (1−p)·τ_t(x), erguido sobre **dois modelos de
resultado**: μ₀ (ajustado só no controle) e μ₁ (só no tratado). Por isso a
seção não para no número: **3.2 a 3.5 abrem a caixa** e mostram de onde o
uplift vem. Um uplift alto demais quase sempre é μ₀ degenerado, não efeito
causal grande.

### 3.1 Ajuste e uplift previsto no holdout

In [4]:
models = uplift.fit_xlearner(train_df, cfg)
uplift_holdout = uplift.predict(models, holdout_df)

por_tipo = (
    uplift_holdout.groupby("offer_type")["uplift"]
    .agg(["mean", "std", "min", "max", "count"])
    .rename(columns={"mean": "uplift_medio", "std": "desvio",
                     "min": "minimo", "max": "maximo", "count": "n"})
)
por_tipo

,uplift_medio,desvio,minimo,maximo,n
offer_type,,,,,
bogo,0.437863,0.284833,-0.066048,1.038681,10206
discount,0.439077,0.334841,-0.158534,1.070360,10206
informational,0.536725,0.212719,-0.020804,1.020288,5057


Um uplift médio dessa magnitude significa "ver a oferta aumenta a
probabilidade de conversão em ~45 pontos percentuais". É grande demais para
um estímulo de marketing. Antes de aceitar, olhe o label dentro de cada braço.

### 3.2 O outcome dentro de cada braço

O X-learner precisa que o outcome seja **observável nos dois braços**. Se o
controle não pode ter `converted=1`, não há o que comparar: μ₀ não tem nenhum
exemplo positivo do qual aprender.

In [5]:
uplift.label_by_arm(holdout_df)

,offer_type,braco,n,outcome_positivo,taxa_outcome
0,bogo,controle (não viu),2366,0,0.000000
1,bogo,tratado (viu),7840,3310,0.422194
2,discount,controle (não viu),3827,0,0.000000
3,discount,tratado (viu),6379,2840,0.445211
4,informational,controle (não viu),1793,0,0.000000
5,informational,tratado (viu),3264,1619,0.496017


`taxa_outcome = 0.000000` no controle, nos três tipos, com **zero** outcomes
positivos em ~21,6 mil linhas. Não é escassez amostral — é **impossibilidade
lógica**: a garantia **G3** do contrato (`specification/schema-processed.md`)
define `converted=1` ⇒ existe `offer viewed`. E `treatment=1` ⇔ viu a oferta.
Logo `treatment=0 ⇒ converted=0` **por construção do label**, não por
comportamento do cliente.

O mesmo vale para `conversion_value` e `reward_cost`, que descendem do mesmo
label influence-aware (Premissa 2). Nenhum outcome do contrato atual admite
valor positivo no controle.

### 3.3 Os estágios do X-learner: μ₀, μ₁ e τ

Se o diagnóstico acima estiver certo, μ₀ deve ser identicamente zero — e a
identidade `τ ≈ μ₁ − μ₀` deve colapsar em `τ ≈ μ₁`.

In [6]:
uplift.stage_diagnostics(models, holdout_df)

,offer_type,mu0_medio,mu0_desvio,mu1_medio,mu1_desvio,mu1_menos_mu0,tau_medio
0,bogo,0.0,0.0,0.438729,0.302277,0.438729,0.437863
1,discount,0.0,0.0,0.439785,0.346977,0.439785,0.439077
2,informational,0.0,0.0,0.536181,0.247825,0.536181,0.536725


`mu0_medio = 0` e `mu0_desvio = 0`: o modelo de controle é a função constante
zero, porque foi ajustado num conjunto onde todo `y` é zero. Logo

$$\tau(x) = \mu_1(x) - \mu_0(x) = \mu_1(x) - 0 = \mu_1(x)$$

O "uplift" **é** a taxa de conversão prevista dos tratados. O modelo não
estima efeito causal nenhum; ele reproduz μ₁ e chama de τ. `tau_medio ≈
mu1_medio` confirma numericamente.

### 3.4 A régua: o teto que um efeito causal teria de respeitar

`src/eda.py::window_spend` mede o gasto na janela de validade **visto ou não
visto** — o único outcome disponível que admite valor no controle. A diferença
bruta entre expostos e não expostos é **confundida** (ver é escolha do
cliente, pós-tratamento: quem abre já era mais ativo), portanto **superestima**
o efeito causal.

A docstring de `eda.naive_spend_lift` já fixava a regra, antes deste notebook
existir: *"se o efeito causal vier maior que esta diferença confundida, é sinal
de erro, não de descoberta."*

In [7]:
events = parse_events(spark, cfg)
offers = read_offers(spark, cfg)
janela = eda.window_spend(attribute(events, offers, cfg), events)

grao = ["account_id", "offer_id", "received_time"]
regua = (
    processed.select(*grao, "treatment")
    .join(janela, on=grao, how="left")
    .fillna({"window_spend": 0.0})
    .groupBy("treatment")
    .agg(
        F.count("*").alias("n"),
        F.mean((F.col("window_spend") > 0).cast("int")).alias("taxa_gastou"),
        F.mean("window_spend").alias("gasto_medio"),
    )
    .orderBy("treatment")
    .toPandas()
)
regua

Premissa 1 violada em 6623 transação(ões): mais de uma oferta ativa no intervalo; prioridade 'earliest_received' aplicada.


,treatment,n,taxa_gastou,gasto_medio
0,0,21623,0.722610,19.669411
1,1,54654,0.811048,30.406769


In [8]:
dif_bruta = regua.loc[regua.treatment == 1, "taxa_gastou"].iloc[0] - \
            regua.loc[regua.treatment == 0, "taxa_gastou"].iloc[0]
uplift_modelo = uplift_holdout["uplift"].mean()

comparacao = pd.DataFrame([
    {"medida": "diferença bruta e CONFUNDIDA (teto)", "pontos_percentuais": 100 * dif_bruta},
    {"medida": "uplift 'estimado' pelo X-learner", "pontos_percentuais": 100 * uplift_modelo},
])
print(comparacao.to_string(index=False))
print(f"\nO modelo reporta {uplift_modelo / dif_bruta:.1f}x o teto bruto — e o teto já superestima.")

                             medida  pontos_percentuais
diferença bruta e CONFUNDIDA (teto)            8.843750
   uplift 'estimado' pelo X-learner           45.797911

O modelo reporta 5.2x o teto bruto — e o teto já superestima.


### 3.5 Veredito

O X-learner reporta um efeito **cerca de 5× maior que o teto bruto não
ajustado** — e o teto já é uma superestimativa, por conter toda a
auto-seleção de quem abre a oferta. Um efeito causal real tem de ficar
*abaixo* dessa régua, não múltiplos acima dela.

A causa é estrutural, não um bug de implementação:

| | |
|---|---|
| **G3** (contrato) | `converted=1` ⇒ houve `offer viewed` |
| **`treatment`** (contrato) | `1` ⇔ viu a oferta |
| **Consequência** | `treatment=0 ⇒ converted=0`, sempre |
| **Efeito no X-learner** | μ₀ ≡ 0 ⇒ τ ≡ μ₁ |

Note que a spec 02 pede, em REQ-202: *"GIVEN um cliente sure thing (μ₀ já
alto) THEN o uplift estimado tende a zero."* Com este label **μ₀ nunca pode
ser alto** — a spec assume um μ₀ que o contrato torna impossível. A Premissa 2
joga justamente o *sure thing* (completou sem ver) para fora do label.

Isto é uma **divergência entre spec e dado**. Pela convenção do projeto
(`CLAUDE.md`), ela se registra aqui e na spec — não se conserta em código por
conta própria: *"o código só muda por decisão de contrato."* As saídas de §4
em diante estão contaminadas por essa degeneração e não devem ser lidas como
resultado causal.

**As duas saídas possíveis**, ambas mudança de contrato:

1. **Tratamento = recebeu** (o que foi de fato aleatorizado, Premissa 4). Mas
   o dataset só contém quem recebeu alguma oferta: não existe controle "não
   recebeu nada", então a comparação vira *entre tipos de oferta*, e a
   Premissa 8 (positividade) já antecipa que "não enviar a ninguém" não é
   avaliável por IPW.
2. **Outcome = gasto na janela** (`window_spend`, visto ou não). Admite μ₀ > 0,
   mas mantém `treatment` = viu, que **não é aleatorizado** — o uplift sai
   confundido pela auto-seleção, e precisaria de premissa causal adicional.

Nenhuma das duas é gratuita, e a escolha é de contrato.

## 4. Avaliação Qini/AUUC

Métrica de uplift, não de classificação (REQ-203): um modelo pode acertar
`converted` e ainda ordenar mal o efeito incremental — Qini mede a segunda
coisa. Calculado no holdout, nunca no treino.

**Leia com a ressalva de §3.5.** Enquanto μ₀ ≡ 0, ordenar por τ é o mesmo que
ordenar por μ₁, e o Qini mede a capacidade de ordenar `converted` — que é
zero em todo o controle. O número abaixo é alto pelo mesmo defeito, não apesar
dele: ele **não** é evidência de que a política terá ganho causal.

In [9]:
avaliacao = holdout_df.merge(
    uplift_holdout[["account_id", "offer_id", "offer_type", "uplift"]],
    on=["account_id", "offer_id", "offer_type"],
)
qini_score = uplift_eval.qini(avaliacao["converted"], avaliacao["uplift"], avaliacao["treatment"])
curva = uplift_eval.qini_curve_points(avaliacao["converted"], avaliacao["uplift"], avaliacao["treatment"])
print(f"Qini AUC (holdout): {qini_score:.4f}  ← inflado por μ₀ ≡ 0, ver §3.5")

Qini AUC (holdout): 0.5477  ← inflado por μ₀ ≡ 0, ver §3.5


In [10]:
uplift_eval.fig_qini_curve(curva, qini_score, cfg, theme="light")

A curva fica muito acima da diagonal — mas isso decorre de `converted=0` em
todo o braço de controle, e não de o modelo ter aprendido *para quem* a oferta
muda o comportamento. Um Qini honesto só sai depois da decisão de contrato de
§3.5.

Mantemos a figura e o número no notebook porque a evidência do defeito é o
próprio número: escondê-lo seria mais fácil e menos verdadeiro.

## Próximo passo

**Bloqueador primeiro.** §3.5 mostra que o uplift atual não é causal. A decisão
de contrato (tratamento ou outcome) precede T-206 — uma política construída
sobre τ ≡ μ₁ alocaria por propensão a converter, que é exatamente a política
de *top-completion* que a spec define como **baseline a ser batido**
(REQ-205), não como a política proposta.

Depois disso, `specification/02-modeling/tasks.md`:

- **T-206** — política sensível a custo: `argmax(uplift_receita − custo)` por
  cliente, incluindo "não enviar".
- **T-207** — baselines de política (aleatória, enviar-a-todos, top-completion).
- **T-208** — avaliação offline por IPW com propensity fixa, checando
  positividade antes de reportar qualquer valor.
- **T-209** — impacto em R$ com intervalo e dimensionamento do próximo A/B.
- **T-210** — SHAP global e force plot individual.

In [11]:
spark.stop()